# 00 — Environment Check & Memory Budget

## Jetson Orin NX Unified Memory Architecture
- **Total Physical Memory**: 16 GB LPDDR5 (shared between CPU and GPU)
- **OS + JupyterLab**: ~4–6 GB
- **Available for experiments**: ~10–12 GB
- **Model weights (Llama-3.2-3B FP16)**: ~6.0 GB
- **KV Cache budget**: ~4–6 GB

⚠️ All GPU memory allocations draw from the same 10–12 GB pool.

In [ ]:
import sys, os, platform
print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Arch    : {platform.machine()}")

In [ ]:
import torch
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version  : {torch.version.cuda}")
    print(f"cuDNN version : {torch.backends.cudnn.version()}")
    dev = torch.cuda.get_device_name(0)
    print(f"Device name   : {dev}")
    prop = torch.cuda.get_device_properties(0)
    total_gb = prop.total_mem / (1024**3)
    print(f"GPU memory    : {total_gb:.1f} GB (unified with CPU)")
    print(f"SM count      : {prop.multi_processor_count}")
    print(f"Compute cap   : {prop.major}.{prop.minor}")
else:
    print("⚠️  CUDA not available — 实验需要 GPU！")

In [ ]:
# JetPack / Jetson version
import subprocess
try:
    jp = subprocess.check_output(['cat', '/etc/nv_tegra_release'], text=True)
    print(f"Tegra release: {jp.strip()}")
except FileNotFoundError:
    print("Not running on Jetson (no /etc/nv_tegra_release)")

# nvcc version
try:
    nvcc = subprocess.check_output(['nvcc', '--version'], text=True)
    for line in nvcc.strip().split('\n'):
        if 'release' in line:
            print(f"nvcc: {line.strip()}")
except FileNotFoundError:
    print("⚠️  nvcc not found — required for KIVI CUDA backend compilation")

In [ ]:
import transformers, datasets
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")

# Check Cache API
from transformers.cache_utils import Cache, DynamicCache
print("Cache API    : OK")

# Check vLLM
try:
    import vllm
    print(f"vLLM         : {vllm.__version__} ✓")
except ImportError:
    print("vLLM         : NOT INSTALLED")
    print("  → PagedAttention experiment requires vLLM")
    print("  → See scripts/setup_vllm_jetson.sh")

In [ ]:
# Unified memory budget check
import shutil

# Disk space (model download needs ~6-10 GB)
usage = shutil.disk_usage("/")
free_gb = usage.free / (1024**3)
print(f"Disk free: {free_gb:.1f} GB")
if free_gb < 10:
    print("⚠️  Less than 10 GB disk space — model download may fail")

# Current memory state
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / (1024**2)
    reserved = torch.cuda.memory_reserved() / (1024**2)
    total = torch.cuda.get_device_properties(0).total_mem / (1024**2)
    print(f"\nGPU Memory (unified):")
    print(f"  Total     : {total:,.0f} MB")
    print(f"  Allocated : {alloc:,.0f} MB")
    print(f"  Reserved  : {reserved:,.0f} MB")
    print(f"  Estimated available: ~{total*0.7:,.0f} MB (after OS + JupyterLab)")
    print("\nTip: Run 'jtop' in a terminal for real-time power and memory monitoring")

In [ ]:
# 验证 src/ 模块导入
import sys
sys.path.insert(0, '..')

from src.metrics import (
    measure_generation, compute_kv_cache_size_mb,
    find_oom_threshold, print_memory_budget,
    JETSON_USABLE_GB,
)
from src.jetson_utils import print_jetson_summary, detect_jetpack_version, is_jetson
from src.kivi_cache import KIVICache
from src.kivi_wrapper import is_cuda_backend_available, get_backend_info
from src.paged_cache import PagedKVCache
from src.vllm_runner import check_vllm_available
from src.dataset_utils import load_prompts
from src.perplexity import compute_perplexity

print("All src modules imported ✓")
print()
print_jetson_summary()
print()
print(f"KIVI backend : {get_backend_info()}")
print(f"vLLM available: {check_vllm_available()}")

---
### Llama-3.2-3B Theoretical KV-Cache Size (GQA vs MHA)

| Parameter | Value |
|-----------|-------|
| num_layers | 28 |
| num_kv_heads (GQA) | 8 |
| head_dim | 128 |
| GQA ratio | 4:1 (32 query heads, 8 KV heads) |

GQA reduces KV Cache size by **4×** compared to standard MHA.

In [ ]:
print(f"{'seq_len':>8} | {'GQA FP16':>10} | {'GQA 2-bit':>10} | {'MHA FP16':>10} | {'ratio':>6}")
print("-" * 60)
for seq_len in [256, 512, 1024, 2048, 4096, 8192]:
    fp16 = compute_kv_cache_size_mb(28, 8, 128, seq_len, dtype_bytes=2)
    q2bit = compute_kv_cache_size_mb(28, 8, 128, seq_len, quant_bits=2, group_size=32)
    mha = compute_kv_cache_size_mb(28, 32, 128, seq_len, dtype_bytes=2)
    ratio = fp16 / q2bit if q2bit > 0 else 0
    print(f"{seq_len:>8} | {fp16:>8.1f} MB | {q2bit:>8.1f} MB | {mha:>8.1f} MB | {ratio:>5.1f}x")

print(f"\n⚠️  KV Cache budget: ~{JETSON_USABLE_GB*1024 - 6000 - 500:.0f} MB (after model weights + system)")